# Biber Daten analysieren

## Daten importieren

Pandas-Modul laden.

In [0]:
import pandas as pd

Dateien einlesen

In [0]:
df = pd.read_csv("../Daten/Biber-Daten-Long.anonym.csv")

oder für komprimierte Rohdaten-Dateien

In [0]:
df = pd.read_csv("../Daten/Biber-Daten-Long.anonym.csv.gz", compression='gzip')

## Überblick verschaffen

In [0]:
df.head()

In [0]:
df.tail(3)

In [0]:
df.describe()

In [0]:
df.info()

## Vorbereitung

Bonuspunkte herausfiltern

In [0]:
f = df['Aufgabe'] != 'Bonus'
df = df[f]

Teams markieren

In [0]:
df["Team"] = df["Teamname"] != "none"

Überprüfen

In [0]:
df.info()

In [0]:
df

## mit groupby oder pivot_table analysieren

In [0]:
df[['Jahr', 'Klassenstufe', 'Punkte']]\
  .groupby(['Jahr', 'Klassenstufe'])\
  .mean()

In [0]:
df[["Jahr", "Klassenstufe", "Schule", "Punkte"]]\
    .groupby(["Jahr", "Schule", "Klassenstufe"])\
    .mean()\
    .sort_values(["Jahr", "Punkte"], ascending=[True, False])

In [0]:
df.pivot_table(
    index = ['Klassenstufe', 'Schule'],
    columns = ['Jahr', 'Team'],
    values='Punkte',
    aggfunc='mean'
)

In [0]:
anzahl = 3
calculate_worst_scores = False

punkte = (
    df.groupby(
        ["Jahr", "Klassenstufe", "Schule", "Klasse", "Name"], as_index=False
    )["Punkte"]
    .sum()
)

punkte["Rang"] = (
    punkte.groupby(["Jahr", "Klassenstufe"])["Punkte"]
    .rank(method="min", ascending=calculate_worst_scores)
)

rangliste = (
    punkte[punkte["Rang"] <= anzahl]
    [["Jahr", "Klassenstufe", "Schule", "Klasse", "Name", "Punkte"]]
    .sort_values(
        ["Jahr", "Klassenstufe", "Punkte", "Klasse", "Name"],
        ascending=[True, False, calculate_worst_scores, True, True]
    )
    .reset_index(drop = True)
)

rangliste.tail(10)

In [0]:
rangliste[rangliste['Jahr'] == 2025]

## Grafiken erstellen

In [0]:
df.plot(kind="box", column="Punkte", by=['Jahr', 'Klassenstufe'], showmeans=True, rot=90)